<div align="center">

# PYTHON · MÓDULO 3
## Pandas + NumPy · Análisis de Datos

**Autor:** Borja Mora Méndez · **Nivel:** Intermedio · **Objetivo:** Data Analyst / BI / Power BI

</div>

---

### ¿De qué trata este archivo?

Aquí empieza lo que de verdad vas a hacer como analista: **manejar tablas de datos con Pandas**.
Si Power BI y SQL son tu mundo, Pandas es ese mismo mundo pero **en código**: cargar datos,
limpiarlos, filtrarlos, agruparlos y sacar conclusiones.

> Idea base: un **DataFrame** es una **tabla de Excel** dentro de Python. Filas, columnas y
> un montón de "botones" (métodos) para transformarla. Y recuerda del Módulo 2: ese DataFrame
> **es un objeto**, así que todo lo que hagas (`df.head()`, `df.groupby()`) son **métodos** suyos.

---

### Hoja de ruta — ¿dónde estás?

```
[x] MÓDULO 1 - Fundamentos
[x] MÓDULO 2 - Programación Orientada a Objetos (POO)
[>] MÓDULO 3 - Pandas + NumPy                          ← ESTÁS AQUÍ
[ ] MÓDULO 4 - Visualización (Matplotlib / Seaborn)
[ ] MÓDULO 5 - Polars + DuckDB
[ ] MÓDULO 6 - Python para Power BI / SQL Server
[ ] MÓDULO 7 - Buenas prácticas profesionales
```

### Índice del Módulo 3
| Parte | Tema |
|-------|------|
| **A — Teoría Pandas** | Series y DataFrame · cargar CSV · `head/tail/info/describe` · limpieza de nulos · columnas y filtros · estadística · `groupby` · `iloc/loc` |
| **B — Proyecto `ventas_500`** | EDA completo · limpieza por criterios · preguntas de negocio · columnas derivadas · percentiles · correlación |

> Tabla de partida en varios ejemplos: `ventas_500.csv` (empleados, ventas, clientes).

---


# PARTE A · Teoría de Pandas

## 1. `Series` y `DataFrame` — las dos piezas base

**Como si tuvieras 10 años:**
- Una **`Series`** es **una sola columna** (una lista con etiquetas).
- Un **`DataFrame`** es **la tabla entera** (muchas columnas juntas).

```
   Series            DataFrame
   ┌────┐        ┌────────┬──────┐
   │ 10 │        │ Nombre │ Edad │
   │ 20 │        ├────────┼──────┤
   │ 30 │        │ Ana    │  23  │
   └────┘        │ Luis   │  35  │
                 │ Marta  │  29  │
                 └────────┴──────┘
```

**Puente:** el DataFrame es tu **tabla de SQL Server** o tu **tabla de Power Query**. La Series
es **una columna** de esa tabla. `import pandas as pd` es el "abrir el programa".

In [ ]:
import pandas as pd

In [ ]:
serie = pd.Series([10,20,30])

df = pd.DataFrame({
 "Nombre":["Ana", "Luis", "Marta"],
 "Edad": [23,35,29]
})

In [ ]:
print(serie)

In [ ]:
print(df)

In [ ]:
display(df)

## 2. Cargar ficheros externos · `read_csv`

Es el **"Obtener datos → CSV"** de Power BI, pero en una línea. `pd.read_csv(ruta)` lee el
archivo y lo convierte en DataFrame.
> La `r` antes de la ruta (`r"C:\..."`) la marca como *raw string*: evita que las barras `\`
> se interpreten como caracteres especiales. Imprescindible en rutas de Windows.

In [ ]:
dp=pd.read_csv(r"ventas_empleados.csv")
display(dp)

## 3. Primeros pasos · conocer la tabla (EDA)

Antes de analizar, **miras** los datos. Estas 4 herramientas son tu primer vistazo:

| Método | Qué hace | Equivalente mental |
|--------|----------|--------------------|
| `.head(n)` | primeras n filas | asomarte por arriba |
| `.tail(n)` | últimas n filas | asomarte por abajo |
| `.info()` | tipos y nulos por columna | la "ficha técnica" |
| `.describe()` | estadísticas (media, min, max…) | resumen numérico |

**EDA = Exploratory Data Analysis.** Es lo primero que hace cualquier analista con una tabla nueva.

In [ ]:
## Visualización de los primeros casos

dp.head(10)

In [ ]:
## Visualización de los últimos casos

dp.tail(10)

In [ ]:
## Información

dp.info()

In [ ]:
## Análisis exploratorio

print(dp.describe())

## 4. Limpieza de datos · valores nulos (`NaN`)

**Como si tuvieras 10 años:** un `NaN` es una **casilla vacía** en la tabla. Antes de hacer
cuentas hay que decidir qué hacer con los huecos: **contarlos**, **borrar** esas filas, o **rellenarlos**.

```
   .isnull().sum()  →  ¿cuántos huecos hay por columna?
   .dropna()        →  BORRA las filas con huecos
   .fillna(valor)   →  RELLENA los huecos con un valor
```

**Puente:** es exactamente "Quitar filas con errores" o "Reemplazar valores" de Power Query.
La decisión (borrar vs rellenar) es una de las más importantes del análisis.

In [ ]:
import numpy as np

df = pd.DataFrame({ "Nombre": ["Ana", "Luis", "Pedro", "Marta", "Juan"], "Edad": [23, np.nan, 35, 28, np.nan], "Ciudad": ["Madrid", "Barcelona", np.nan, "Valencia", "Sevilla"], "Ventas": [200, 150, np.nan, 300, 250] })


display(df)

In [ ]:
## Contar el numero de valores nulos para cada columna

df.isnull().sum()

In [ ]:
## Eliminar filas que contengan un valor nulo

df_sin_nulos = df.dropna()
display(df_sin_nulos)

In [ ]:
## Rellenar los nulos por ceros

df_relleno = df.fillna(0)
display(df_relleno)
display(df)

## 5. Crear columnas nuevas y filtrar filas

**Columna nueva:** `df["Ingresos"] = df["Unidades"] * df["Precio"]`. Es como una **columna
calculada** de Power BI/DAX, pero operando sobre toda la columna de golpe (*vectorizado*).

**Filtrar:** `df[df["Producto"] == "Laptop"]` se lee de dentro afuera: primero la condición
(devuelve True/False por fila), y el DataFrame se queda **solo con las filas True**.

```
   df[ df["Unidades"] > 4 ]
        └── máscara de True/False ──┘
        →  solo pasan las filas donde es True
```

**Puente:** la columna nueva es una *calculated column*; el filtro es el `WHERE` de SQL o el
filtro de un visual en Power BI.

In [ ]:
## Nuevo dataframe

df = pd.DataFrame({"Fecha": pd.to_datetime(["2025-01-01","2025-01-02","2025-01-03","2025-01-04","2025-01-05"]),
 "Producto": ["Laptop","Tablet","Smartphone","Laptop","Tablet"],
 "Unidades": [5, np.nan, 8, 3, 6],
 "Precio": [800, 300, np.nan, 820, 310] })


In [ ]:
## Crear una columna nueva

df["Ingresos"] = df["Unidades"] * df["Precio"]
display(df)

In [ ]:
## Filtrado de datos

df[df["Producto"] == "Laptop"]

In [ ]:
df[df["Unidades"] > 4]

## 6. Estadística básica y ordenación

`.mean()`, `.max()`… aplican una cuenta a **toda una columna**. Y un truco muy útil:
**rellenar los huecos con la media** de la propia columna (`fillna(df["x"].mean())`), para no
perder filas pero tampoco inventar.

`.sort_values(by="Ingresos", ascending=False)` = **ordenar de mayor a menor**, como pinchar la
cabecera de una columna en Excel.

**Puente:** `.mean()` = `AVG()` en SQL / `AVERAGE()` en DAX. `.sort_values()` = `ORDER BY`.

In [ ]:
df["Unidades"].mean()


In [ ]:
df["Unidades"].max()

In [ ]:
#€ Rellenar nulos con la media

df["Unidades"] = df["Unidades"].fillna(df["Unidades"].mean())
df["Precio"] = df["Precio"].fillna(df["Precio"].mean())

print(df)

In [ ]:
df.sort_values(by="Ingresos", ascending=False)

## 7. Agrupaciones · `groupby` (el corazón del análisis)

**Como si tuvieras 10 años:** `groupby` es **hacer montones**. Juntas las filas que comparten
algo (ej. el mismo producto) y sacas un número por montón (la suma, la media…).

```
   df.groupby("Producto")["Ingresos"].sum()
        │                    │          └── ¿qué cuenta? → la suma
        │                    └── ¿de qué columna?
        └── ¿por qué agrupamos?
```

**Puente directísimo:** es el `GROUP BY` de SQL, o lo que hace una **matriz/tabla dinámica**
en Power BI al arrastrar una dimensión y una medida. Si dominas esto, dominas Pandas.

In [ ]:
## Ingresos por producto


df.groupby("Producto")["Ingresos"].sum()

In [ ]:
## Promedio de precio por producto


promedio_precio_por_producto = df.groupby("Producto")["Precio"].mean()
print(promedio_precio_por_producto)

## 8. Acceder a filas concretas · `iloc` y `loc`

Dos formas de señalar filas:
- **`.iloc[n]`** → por **posición** (el número de asiento, empieza en 0). *i = index numérico.*
- **`.loc[etiqueta]`** → por **etiqueta/condición** (por nombre o por una regla).

Un combo precioso: `df.loc[df["Ingresos"].idxmax()]` → `idxmax()` te da la **posición del valor
máximo**, y `.loc[...]` trae esa fila entera. Es "dame la fila del campeón".

**Puente:** `iloc` es "la fila número N"; `loc` con condición es como un `WHERE` que devuelve
el registro top. `idxmax()` ≈ `TOPN(1, ...)` en DAX.

In [ ]:
display(df)

In [ ]:
df.iloc[4]

In [ ]:
df.iloc[1:4]

In [ ]:
## Te devuelve el máximo de la columna que le metamos
df.loc[df["Ingresos"].idxmax()]

---
# PARTE B · Proyecto real · `ventas_500`

Ahora aplicas **todo lo anterior** a un caso de empresa: ventas de empleados por departamento.
Este es el tipo de mini-proyecto que enseñarías en una entrevista de Data Analyst.

**Flujo profesional que vas a seguir:**
```
   Cargar  →  Explorar (EDA)  →  Limpiar  →  Responder preguntas de negocio  →  Correlación
```

## B.1 · Planteamiento e importación
El comentario inicial define el **objetivo de negocio** (buena costumbre: documentar antes de codear).

In [ ]:
# De ventas de los empleados de una empresa

# Se dispone de un fichero con información de:


#Nombre del empleado
#Edad
#Departamento
#Ventas mensuales
#Numero de clientes atendidos


#Objetivo


# Cargar los datos en Python
# Limpiar datos incorrectos o vacios
# Analizar estadisticas básicas
# Realizar cálculos con NumPy
# Realizar preguntas sobre el negocio


In [ ]:
#PASO 1 - Importar librerias

import pandas as pd
import numpy as np

In [ ]:
#PASO2 - Cargar archivo

dp=pd.read_csv(r"ventas_empleados.csv")
display(dp)

## B.2 · EDA — radiografía del dataset
Aquí brillan dos cálculos muy "de informe": el **total de nulos** de un vistazo
(`.isna().sum().sum()` — doble suma: por columna y luego total) y el **% de nulos por columna**,
que es justo lo que pondrías en un informe de calidad de datos.

In [ ]:
#PASO 3 - EDA

dp.info()

In [ ]:
# Total de nulos en TODO el dataset, un solo número
dp.isna().sum().sum()

In [ ]:
# Porcentaje de nulos por columna (muy útil en informes)
(dp.isna().sum() / len(dp) * 100).round(2)


In [ ]:
dp.describe()

## B.3 · Limpieza (opción A): eliminar filas con nulos
`dropna()` crea una versión sin huecos. Comparas el `describe()` del original con el limpio para
ver **cuánto cambia** al borrar.
> Detalle menor: en los `print("\ Archivo limpio", ...)` esa `\ ` es una secuencia de escape
> rara (probablemente querías `\n`). Funciona, pero Python puede avisar. Lo dejo tal cual.

In [ ]:
#PASO 4 - Limpieza de datos

#Eliminar filas con valores nulos


dp_limpio =dp.dropna()
display(dp_limpio)
dp_limpio.describe()

In [ ]:
print("\ Archivo limpio",
 dp.describe())

In [ ]:
print("\ Archivo sin Na",
 dp_limpio.describe())

## B.4 · Limpieza (opción B): rellenar por criterio
Más fino que borrar: rellenas **cada columna con la lógica que tiene sentido** — la edad con su
media, las ventas/clientes con 0 (porque un nulo ahí seguramente significa "no vendió").

In [ ]:
### Rellenar valores vacios por variables segun diferentes criterios

dp["edad_limpio"]=dp["edad"].fillna(dp["edad"].mean())
dp["ventas_limpio"]=dp["ventas"].fillna(0)
dp["clientes_limpio"]=dp["clientes"].fillna(0)

In [ ]:
# Mostrar filas con ventas NaN
dp[dp["ventas"].isna()]

## B.5 · Preguntas de negocio
La parte que de verdad importa: convertir la tabla en **respuestas**. Media de ventas, ventas por
departamento, el empleado top (`idxmax`), mayores de 30, qué departamento vende más…
Cada una de estas líneas es un `GROUP BY` / `WHERE` / `TOPN` que ya harías en SQL o Power BI.

In [ ]:
# Calcular la media de las ventas

media_ventas = np.mean(dp["ventas"])
print(media_ventas)

In [ ]:
## ventas por departamento

ventas_dep = dp.groupby('departamento')['ventas'].mean()
print(ventas_dep)

In [ ]:
# empleado con más venta

top = dp.loc[dp['ventas'].idxmax()]
print(top)

In [ ]:
#Listado de los nombres de los empleados

listado = dp["nombre"]

print(listado)

In [ ]:
#Empleados mayores de 30 años

print(dp[dp['edad']> 30])

In [ ]:
#Media de edad

print(dp['edad'].mean())

In [ ]:
#Que departamento genera más ventas

ventas_dp = dp.groupby('departamento')['ventas'].sum()
print(ventas_dp)

In [ ]:
ventas_dp = dp.groupby('departamento')['ventas'].sum()
print(ventas_dp.idxmax(), max(ventas_dp))

## B.6 · Columnas derivadas y el problema de dividir entre 0
Creas `ventas_x_cliente = ventas / clientes`. Pero si un empleado tiene **0 clientes**, dividir
entre 0 rompe el cálculo (da infinito o `NaN`). La solución elegante: `.replace(0, np.nan)` para
convertir esos ceros en nulos antes de dividir.

Este es un error **clásico y muy profesional** de detectar. Lo viste venir tú solo.

In [ ]:
#Nueva columna para ver las ventas por cliente = ventas/cliente

dp["ventas_x_cliente"]=dp["ventas"]/dp["clientes"]
display(dp)

In [ ]:
#Problema cuando hay valores 0 y nos rompe la división

dp["ventas_x_cliente"]=dp["ventas"]/dp["clientes"]
display(dp)

dp["ventas_x_cliente_d0"]=dp["ventas"]/dp["clientes"].replace(0,np.nan)
display(dp)


## B.7 · Percentiles
Un **percentil** parte los datos en cien trozos. El percentil 90 de ventas = el valor por debajo
del cual está el 90% de la gente; por encima queda el **10% que más vende**.
> Ojo al etiquetado: la variable se llama `p10` y `ventas_bajas`, pero al usar `percentile(...,90)`
> y filtrar `> p10` en realidad estás sacando el **10% más ALTO**. El cálculo es correcto; solo el
> nombre despista. Buen punto para repasar.

In [ ]:

#Quiero sacar el percentil X de las ventas

p10= np.percentile(dp["ventas"].dropna(),90)
ventas_bajas=dp[dp["ventas"]>p10]
print(ventas_bajas)


In [ ]:
print(len(ventas_bajas))

In [ ]:
ventas_bajas =dp[dp["ventas"]>510]
print(ventas_bajas)
print(len(ventas_bajas))

## B.8 · Correlación
La **correlación** mide si dos variables se mueven juntas (de −1 a +1). `df.corr()` te da la
**matriz de correlación** de todas las columnas numéricas.
> En el último ejemplo `["edad","clientes","edad"]` repite "edad" — la matriz sale igualmente,
> pero con una fila/columna duplicada. Curiosidad inofensiva.

**Puente:** en Power BI esto suele verse como un *scatter* con línea de tendencia; aquí lo tienes
en números, listo para decidir qué relaciones merece la pena visualizar (¡eso es el Módulo 4!).

In [ ]:
print("\n ***** Matriz de correlación*****")
print(dp.corr(numeric_only=True))

In [ ]:
variables=["edad","ventas"]
correlac=dp[variables].corr()
print(correlac)

In [ ]:
variables=["edad","clientes"]
correlac=dp[variables].corr()
print(correlac)

In [ ]:
variables=["edad","clientes", "edad"]
correlac=dp[variables].corr()
print(correlac)

---
## Resumen del Módulo 3 — lo que ya dominas

```
Series (columna) vs DataFrame (tabla)
Cargar CSV con read_csv (rutas raw r"...")
EDA: head, tail, info, describe
Calidad de datos: isna().sum(), % de nulos
Limpieza: dropna (borrar) vs fillna (rellenar por criterio)
Columnas calculadas (vectorizadas) y filtros con máscaras
Estadística: mean, max, sort_values
groupby (= GROUP BY) → el corazón del análisis
Acceso a filas: iloc (posición) y loc (etiqueta/condición) + idxmax
División por cero con replace(0, np.nan)
Percentiles y matriz de correlación
```

### Próximo paso
**Módulo 4 — Visualización (Matplotlib + Seaborn).** Pasamos de los números a los **gráficos**:
la correlación que acabas de calcular se convierte en un mapa de calor, y las distribuciones en
histogramas. Es donde el análisis se vuelve **comunicación**.

> **Tres ideas clave:** (1) DataFrame = tabla de Excel en código. (2) `groupby` = `GROUP BY`.
> (3) Limpiar bien los datos es el 80% del trabajo real del analista.
